<a href="https://colab.research.google.com/github/gugomesx10/CP2-GEN_AI/blob/main/CP2_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHECKPOINT 02: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Gustavo Gomes Martins - RM555999
- Matheus de Mattos Vecchi - RM561716
- Nicholas Albuquerque Buzo	- RM561082
- Nicholas Camillo Canadas de Paula - RM561262

**Tema escolhido:** Suporte Técnico de IOT

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória e troca de modelo
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [16]:
!pip install google-genai gradio -q

In [15]:
from google import genai
from google.genai import types
from google.colab import userdata

In [17]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

GEMINI_API_KEY ok


In [22]:
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [19]:
# >>> PERSONALIZE: system prompt do tema do grupo.
# Um bom system diz: quem o assistente é, sobre o que fala,
# como responde (idioma, tamanho) e o que não deve fazer.
SYSTEM_PROMPT = """Você é um assistente especializado em suporte técnico de IoT.

Sua função é ajudar usuários a diagnosticar problemas com ESP32, Arduino, sensores, redes Wi-Fi e dispositivos conectados.

Forneça orientações técnicas claras, organizadas e adequadas para iniciantes.

Responda sempre em português, utilizando no máximo 5 frases.

Se a pergunta não estiver relacionada a IoT, informe educadamente que o assunto está fora da sua especialidade.

Não invente especificações técnicas e não recomende procedimentos elétricos perigosos."""

---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [20]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [23]:
# >>> PERSONALIZE: Etapa 1 - Assistente com Guardrails

versoes_system = {
    "Sem restrição": """Você é um assistente prestativo.
Responda sempre em português.""",

    "Com restrição de tema": """Você é um especialista em suporte técnico de IoT.
Responda apenas perguntas relacionadas a ESP32, Arduino,
sensores, redes Wi-Fi e dispositivos IoT.
Se a pergunta estiver fora desse tema, recuse educadamente.
Responda sempre em português.""",

    "Com restrição de tema e formato": """Você é um especialista em suporte técnico de IoT.
Responda apenas perguntas relacionadas a ESP32, Arduino,
sensores, redes Wi-Fi e dispositivos IoT.
Se a pergunta estiver fora desse tema, recuse educadamente.
Responda sempre em português, em no máximo 3 frases,
sem utilizar listas ou tópicos."""
}

# Perguntas para testar os guardrails
pergunta_dentro = "Por que meu ESP32 não consegue conectar ao Wi-Fi?"
pergunta_fora = "Qual é a capital da França?"

# Executa os testes utilizando apenas o Google Gemini
for nome, system in versoes_system.items():
    print(f"\n{'=' * 55}")
    print(f"  {nome}")
    print(f"{'=' * 55}")

    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_gemini(pergunta_dentro, system))

    print(f"\n[Fora do escopo] {pergunta_fora}")
    print(perguntar_gemini(pergunta_fora, system))

print("\nTeste de guardrails finalizado!")


  Sem restrição

[Dentro do escopo] Por que meu ESP32 não consegue conectar ao Wi-Fi?
Existem várias razões pelas quais o ESP32 pode falhar ao conectar-se ao Wi-Fi. Como o ESP32 é sensível a alguns detalhes de configuração e hardware, o problema geralmente se resume a uma das seguintes causas comuns. 

Aqui está um passo a passo para diagnosticar e resolver o problema:

### 1. SSID ou Senha Incorretos
Este é o erro mais comum. 
* **Verifique maiúsculas e minúsculas:** O ESP32 diferencia letras maiúsculas de minúsculas tanto no nome da rede (SSID) quanto na senha.
* **Redes ocultas:** Se o seu roteador oculta o SSID, você precisa configurar explicitamente o ESP32 para se conectar a redes ocultas (dependendo da biblioteca que estiver usando).

### 2. Problemas com a Frequência (2.4 GHz vs 5 GHz)
* **O ESP32 só suporta Wi-Fi de 2.4 GHz.** Ele **não** consegue se conectar a redes de 5 GHz.
* Se o seu roteador usa "Smart Connect" (mesmo nome para as redes de 2.4 GHz e 5 GHz), o ESP32 pode 

### Observações do grupo (Etapa 1)

**1. O assistente sem restrição respondeu à pergunta fora do escopo?**

Sim. O Gemini respondeu normalmente que a capital da França é Paris, mesmo sendo uma pergunta fora do tema de IoT. Isso aconteceu porque não havia nenhuma restrição de assunto no system prompt.

**2. Com o guardrail de tema, ele recusou? A recusa foi educada?**

Sim. O Gemini recusou educadamente a pergunta sobre a França, explicando que sua especialidade é suporte técnico de IoT. Também ofereceu ajuda com ESP32, Arduino, sensores e redes Wi-Fi, respeitando o tema definido.

**3. A restrição de formato mudou o estilo da resposta?**

Sim. Sem a restrição de formato, o Gemini apresentou respostas mais longas, utilizando listas e tópicos. Com a restrição, as respostas ficaram mais curtas e objetivas, respeitando o limite de três frases e sem utilizar listas.

---
## Etapa 2: Comparação com temperatura

Na atividade anterior o grupo comparou Hugging Face e Gemini com o mesmo prompt.
Aqui o grupo compara o efeito da **temperatura** nos dois modelos.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

A mesma pergunta é enviada com cada valor de temperatura para os dois modelos.

In [24]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [25]:
# Etapa 2 - Comparação de Temperaturas
# Tema: Suporte Técnico de IoT

perguntas = [
    "Quais são as principais causas de falha na leitura de um sensor DHT22 conectado ao ESP32?",
    "Como melhorar a segurança da comunicação Wi-Fi entre dispositivos IoT?"
]

temperaturas = [0.0, 1.0]

for pergunta in perguntas:
    print("\n" + "=" * 60)
    print(f"PERGUNTA: {pergunta}")
    print("=" * 60)

    for temperatura in temperaturas:
        print(f"\n--- Gemini | Temperatura: {temperatura} ---")

        resposta = perguntar_gemini(
            pergunta,
            SYSTEM_PROMPT,
            temperatura=temperatura
        )

        print(resposta)
        print("-" * 60)

print("\nComparação de temperaturas finalizada!")


PERGUNTA: Quais são as principais causas de falha na leitura de um sensor DHT22 conectado ao ESP32?

--- Gemini | Temperatura: 0.0 ---
As principais causas de falha na leitura do sensor DHT22 com o ESP32 são problemas na fiação, alimentação instável e ausência de resistor de pull-up. Verifique se o pino de dados está conectado à porta GPIO correta configurada no seu código e se a alimentação de 3,3V está firme. O uso de cabos muito longos também pode causar perda de sinal na comunicação de um único fio. Por fim, certifique-se de respeitar o intervalo mínimo de dois segundos entre as leituras exigido pelo sensor.
------------------------------------------------------------

--- Gemini | Temperatura: 1.0 ---
As falhas na leitura do sensor DHT22 no ESP32 geralmente ocorrem por problemas na conexão física, como cabos invertidos ou mal conectados. Outra causa comum é a ausência de um resistor de pull-up (geralmente de 4,7kΩ a 10kΩ) entre o pino de dados e a alimentação de 3,3V. Problemas d

### Observações do grupo (Etapa 2)

**1. Com `temperature=0.0`, as respostas ficaram mais consistentes?**

Com a temperatura 0.0, o Gemini apresentou respostas técnicas objetivas sobre problemas no sensor DHT22 e segurança em redes IoT. As informações foram organizadas de forma clara, mas seriam necessários testes repetidos para confirmar uma maior consistência.

**2. Com `temperature=1.0`, as respostas ficaram mais variadas?**

Sim, observamos pequenas variações na linguagem e nos detalhes técnicos. Na pergunta sobre o DHT22, por exemplo, o Gemini apresentou valores específicos para o resistor de pull-up. Apesar disso, as respostas mantiveram informações semelhantes às geradas com temperatura 0.0.

**3. Qual combinação de modelo e temperatura consideramos melhor para o tema?**

Consideramos o Gemini 3.5 Flash-Lite com temperatura 0.0 a configuração mais adequada para suporte técnico de IoT. Como o objetivo é orientar diagnósticos de ESP32, Arduino e sensores, priorizamos respostas objetivas e previsíveis, em vez de maior criatividade.

**Observação:** A comparação entre dois modelos foi substituída por testes de diferentes temperaturas no Gemini, conforme a orientação atualizada do professor.

---
## Etapa 3: Chat com memória e troca de modelo

Na atividade anterior o chat usava apenas o Hugging Face.
Aqui o grupo pode trocar o modelo durante a conversa, mantendo o histórico.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas
- `modelo hf` — troca para Hugging Face
- `modelo gemini` — troca para Gemini

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", troque o modelo e pergunte de novo.

In [27]:
# Etapa 3 - Chat com memória usando Google Gemini

def responder_chat(historico):
    conteudos = []

    for msg in historico:
        if msg["role"] == "system":
            continue

        papel = "model" if msg["role"] == "assistant" else "user"

        conteudos.append({
            "role": papel,
            "parts": [{"text": msg["content"]}]
        })

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=0.7
        ),
    )

    return resposta.text

In [28]:
# Etapa 3 - Chat com memória usando Google Gemini
# Tema: Suporte Técnico de IoT

historico = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

print("Chat iniciado com Google Gemini!")
print("Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ").strip()

    if entrada.lower() == "sair":
        print("\nEncerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]
        print("\nHistórico apagado!\n")
        continue

    if entrada.lower() == "historico":
        print(f"\nMensagens armazenadas: {len(historico)}\n")
        continue

    if not entrada:
        print("Digite uma mensagem válida.")
        continue

    historico.append({
        "role": "user",
        "content": entrada
    })

    try:
        texto = responder_chat(historico)

        historico.append({
            "role": "assistant",
            "content": texto
        })

        print(f"\n[Gemini] {texto}\n")

    except Exception as erro:
        historico.pop()
        print(f"\nErro ao consultar Gemini: {erro}\n")

Chat iniciado com Google Gemini!
Comandos: sair | limpar | historico

Você: Meu nome é Gustavo e estou trabalhando com um ESP32.

[Gemini] Olá, Gustavo! É um prazer ajudar você nos seus projetos com o ESP32. Por favor, descreva qual é o problema ou dúvida técnica que você está enfrentando no momento, como falhas de conexão Wi-Fi, leitura de sensores ou gravação de código. Estou à disposição para orientá-lo passo a passo na solução!

Você: Qual é o meu nome e qual dispositivo estou utilizando?

[Gemini] O seu nome é Gustavo e você está utilizando um microcontrolador ESP32. Como assistente técnico de IoT, estou pronto para ajudar com qualquer dúvida ou problema relacionado ao seu dispositivo. Basta me dizer qual é o seu próximo desafio no projeto!

Você: historico

Mensagens armazenadas: 5

Você: limpar

Histórico apagado!

Você: Qual é o meu nome e qual dispositivo estou utilizando?

[Gemini] Como sou um assistente de suporte técnico de IoT, não tenho acesso a informações pessoais como 

In [29]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um assistente especializado em suporte técnico de IoT.  Sua função é ajud
[user] Qual é o meu nome e qual dispositivo estou utilizando?
[assistant] Como sou um assistente de suporte técnico de IoT, não tenho acesso a informações


### Observações do grupo (Etapa 3)

**1. Ao trocar de modelo no meio da conversa, o novo modelo "lembrou" do que foi dito?**

Como utilizamos apenas o Google Gemini 3.5 Flash-Lite, não realizamos a troca entre modelos. Porém, testamos a memória do assistente informando que o usuário se chamava Gustavo e estava utilizando um ESP32. O Gemini recuperou corretamente essas informações. Após executar o comando `limpar`, ele não conseguiu mais identificar o nome nem o dispositivo.

**2. Por que isso acontece (ou não acontece)?**

Isso acontece porque o histórico da conversa é armazenado em uma lista Python chamada `historico`. A cada nova pergunta, as mensagens anteriores são enviadas novamente ao Gemini, permitindo que ele utilize o contexto. Quando executamos o comando `limpar`, essa lista é reiniciada, fazendo com que o assistente deixe de receber as informações anteriores.

**3. O que muda na forma como o histórico é enviado para o Gemini em relação ao Hugging Face?**

No Hugging Face, o histórico utiliza mensagens com os papéis `system`, `user` e `assistant`. No Gemini, as mensagens da conversa utilizam `user` e `model`, com o conteúdo organizado em `parts`. Além disso, as instruções do sistema são enviadas separadamente pelo parâmetro `system_instruction`. Nossa implementação realiza essa conversão antes de enviar as mensagens ao Gemini.

**Observação:** Conforme a orientação atualizada do professor, adaptamos a etapa para demonstrar a memória e a limpeza do histórico utilizando exclusivamente o Google Gemini.

---
## Etapa 4: Interface web com Gradio e controle de temperatura

Na atividade anterior a interface tinha o seletor de modelo.
Aqui o grupo adiciona um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [30]:
# Etapa 4 - Interface Gradio com Google Gemini
# Tema: Suporte Técnico de IoT

import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo

    if isinstance(conteudo, list):
        return " ".join(
            item.get("text", "")
            for item in conteudo
            if isinstance(item, dict)
        )

    return str(conteudo)


def responder(mensagem, historico_gradio, temperatura):
    conteudos = []

    # Recupera o histórico da conversa
    for msg in historico_gradio:
        papel = "model" if msg["role"] == "assistant" else "user"

        conteudos.append({
            "role": papel,
            "parts": [{
                "text": texto_da_mensagem(msg["content"])
            }]
        })

    # Adiciona a pergunta atual
    conteudos.append({
        "role": "user",
        "parts": [{"text": mensagem}]
    })

    # Envia a conversa para o Gemini
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=temperatura
        )
    )

    return resposta.text

In [31]:
# Etapa 4 - Interface Web com Gradio
# Tema: Suporte Técnico de IoT

gr.ChatInterface(
    fn=responder,

    additional_inputs=[
        gr.Slider(
            minimum=0.0,
            maximum=1.0,
            step=0.1,
            value=0.7,
            label="Temperatura"
        )
    ],

    title="IoT Tech Support | Assistente Inteligente",

    description=(
        "Assistente técnico especializado em ESP32, Arduino, "
        "sensores, redes Wi-Fi e dispositivos IoT. "
        "Descreva seu problema e receba orientações técnicas "
        "utilizando inteligência artificial Google Gemini."
    ),

    examples=[
        ["Por que meu ESP32 não conecta ao Wi-Fi?", 0.0],
        ["Como conectar um sensor DHT22 ao ESP32?", 0.3],
        ["Como funciona o protocolo MQTT?", 0.7],
        ["Como diagnosticar falhas em sensores IoT?", 1.0]
    ],

).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1eabb2c1ab57f3d500.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Observações do grupo (Etapa 4)

**1. A interface foi testada e o print foi registrado?**

Sim. Desenvolvemos uma interface web utilizando Gradio, integrada ao Google Gemini 3.5 Flash-Lite. A interface apresenta um chat para suporte técnico de IoT e um slider para controlar a temperatura das respostas entre 0.0 e 1.0. Realizamos os testes e registramos uma captura de tela para incluir no README do repositório.

**2. Ao testar a mesma pergunta com temperatura 0.0 e 1.0, o que mudou?**

Utilizamos a pergunta "Por que meu ESP32 não consegue conectar ao Wi-Fi?" nas duas temperaturas. Em ambos os testes, o Gemini apresentou possíveis causas para o problema, como credenciais incorretas, incompatibilidade de frequência Wi-Fi e falhas de alimentação. Na temperatura 1.0, observamos pequenas variações na linguagem e na organização das informações, mas as orientações técnicas permaneceram semelhantes.

**3. Qual combinação de modelo e temperatura funcionou melhor para o tema?**

Consideramos o Gemini 3.5 Flash-Lite com temperatura 0.0 a configuração mais adequada para nosso assistente de suporte técnico de IoT. Essa configuração prioriza respostas previsíveis e objetivas, características importantes para orientar o diagnóstico de problemas com ESP32, Arduino, sensores e redes Wi-Fi.

**Observação:** Conforme a orientação do professor, utilizamos exclusivamente o Google Gemini, mantendo o controle de temperatura como recurso principal da interface.

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [32]:
%%writefile app_avaliacao.py

import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

# Configuração do Google Gemini
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

SYSTEM_PROMPT = os.environ.get(
    "SYSTEM_PROMPT",
    """Você é um assistente especializado em suporte técnico de IoT.
Ajude usuários com problemas de ESP32, Arduino, sensores,
redes Wi-Fi e dispositivos conectados.
Responda em português, em no máximo 5 frases.
Recuse educadamente perguntas fora do tema."""
)

# Inicialização da API
app = FastAPI(
    title="IoT Tech Support API",
    description="API de suporte técnico de IoT com Google Gemini",
    version="1.0.0"
)

# Histórico independente para cada sessão
sessoes: dict[str, list] = {}


class Pergunta(BaseModel):
    session_id: str
    mensagem: str


@app.post("/chat")
def chat(pergunta: Pergunta):

    session_id = pergunta.session_id

    # Recupera o histórico da sessão
    historico = sessoes.get(session_id, [])

    # Prepara as mensagens para o Gemini
    conteudos = historico + [
        {
            "role": "user",
            "parts": [{"text": pergunta.mensagem}]
        }
    ]

    # Consulta o Gemini
    resposta = cliente.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=0.7
        )
    )

    texto = resposta.text

    if not texto:
        raise ValueError("O Gemini não retornou uma resposta de texto.")

    # Salva o histórico somente após a resposta
    sessoes[session_id] = conteudos + [
        {
            "role": "model",
            "parts": [{"text": texto}]
        }
    ]

    return {
        "session_id": session_id,
        "resposta": texto
    }


@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):

    sessoes.pop(session_id, None)

    return {
        "status": "sessão apagada",
        "session_id": session_id
    }

Writing app_avaliacao.py


In [33]:
# Instala as dependências da API
!pip install fastapi uvicorn -q

import os
import subprocess
import time

# Configura as variáveis de ambiente
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

# Inicia o servidor FastAPI
servidor = subprocess.Popen([
    "uvicorn",
    "app_avaliacao:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

# Aguarda a inicialização
time.sleep(5)

if servidor.poll() is None:
    print("Servidor iniciado. Verifique os logs e teste a API.")
    print("Endereço: http://localhost:8000")
else:
    print("Erro: o servidor foi encerrado durante a inicialização.")

Servidor iniciado. Verifique os logs e teste a API.
Endereço: http://localhost:8000


In [35]:
# Etapa 5 - Teste de sessões independentes

import requests

URL = "http://localhost:8000/chat"

# Sessão A - Apresenta informações
requests.post(
    URL,
    json={
        "session_id": "sessao_a",
        "mensagem": "Meu nome é Gustavo e estou desenvolvendo um projeto IoT com ESP32."
    },
    timeout=60
).raise_for_status()

# Sessão A - Verifica a memória
r_a = requests.post(
    URL,
    json={
        "session_id": "sessao_a",
        "mensagem": "Qual é o meu nome e qual dispositivo IoT estou utilizando?"
    },
    timeout=60
)
r_a.raise_for_status()

# Sessão B - Não recebeu informações anteriores
r_b = requests.post(
    URL,
    json={
        "session_id": "sessao_b",
        "mensagem": "Qual é o meu nome e qual dispositivo IoT estou utilizando?"
    },
    timeout=60
)
r_b.raise_for_status()

print("=" * 50)
print("TESTE DE SESSÕES INDEPENDENTES")
print("=" * 50)

print("\nSessão A:")
print(r_a.json()["resposta"])

print("\nSessão B:")
print(r_b.json()["resposta"])

print("\nTeste finalizado!")

TESTE DE SESSÕES INDEPENDENTES

Sessão A:
Seu nome é Gustavo e o dispositivo IoT que você está utilizando é o ESP32. Como posso ajudar com o seu projeto hoje?

Sessão B:
Como mencionei anteriormente, não tenho acesso a informações pessoais como o seu nome. 

Também não consigo visualizar automaticamente o hardware que está na sua bancada. 

Para que eu possa ajudar com o diagnóstico, por favor, me diga qual microcontrolador e quais sensores você está usando.

Teste finalizado!


In [36]:
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.


### Observações do grupo (Etapa 5 — Bônus)

**1. A sessão A respondeu o nome corretamente? E a sessão B?**

No teste, a sessão A recebeu a informação de que a usuária se chamava Ana e estava utilizando um ESP32. O resultado esperado é que o Gemini consiga recuperar essas informações. Já a sessão B, por não ter recebido esses dados anteriormente, deve informar que desconhece o nome e o dispositivo utilizado.

**2. O que mudou nessa API em relação à da atividade anterior?**

Na atividade anterior, a API respondia às perguntas sem manter o histórico entre as requisições. Nesta implementação, utilizamos um dicionário Python para armazenar conversas separadas por `session_id`. Assim, cada sessão mantém seu próprio contexto, permitindo conversas independentes com o Google Gemini.

**3. Cite um caso de uso real onde sessões independentes seriam necessárias.**

Um exemplo seria uma plataforma de suporte técnico de IoT utilizada por diferentes clientes. Enquanto um usuário solicita ajuda com um ESP32 que não conecta ao Wi-Fi, outro pode consultar problemas de leitura de um sensor DHT22. Cada atendimento precisa manter seu próprio histórico, evitando que informações de usuários diferentes sejam misturadas.